In [1]:
import pandas as pd
import numpy as np

In [2]:
# =========================================================
# STEP 1: IMPORT DATASET
# =========================================================
df = pd.read_csv("cybersecurity_cases_india_combined.csv")
print("Original Shape :", df.shape)
df.head()
df.info()

Original Shape : (1200, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Year             1200 non-null   int64 
 1   Day              1200 non-null   int64 
 2   Amount_Lost_INR  1200 non-null   int64 
 3   Incident_Type    1200 non-null   object
 4   City             1200 non-null   object
 5   Category         1200 non-null   object
dtypes: int64(3), object(3)
memory usage: 56.4+ KB


In [3]:
# =========================================================
# STEP 2: RENAME COLUMNS FOR SQL / POWER BI
# =========================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Columns after rename:")
print(df.columns.tolist())

Columns after rename:
['year', 'day', 'amount_lost_inr', 'incident_type', 'city', 'category']


In [4]:
# =========================================================
# STEP 3: REMOVE DUPLICATES
# =========================================================

duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

df = df.drop_duplicates().reset_index(drop=True)

Duplicate rows: 0


In [5]:
# =========================================================
# STEP 4: CLEAN TEXT COLUMNS
# =========================================================

text_columns = df.select_dtypes(include=["object"]).columns

for col in text_columns:
    df[col] = df[col].str.strip().str.title()


In [6]:
# =========================================================
# STEP 5: FIX KNOWN SPELLING / CATEGORY ISSUES
# =========================================================

if "incident_type" in df.columns:
    df["incident_type"] = df["incident_type"].replace({
        "Malware_Attacks": "Malware",
        "Malware Attack": "Malware",
        "Phising": "Phishing",
        "Cyber Fraud": "Online Fraud",
        "Financial Fraud": "Online Fraud"
    })

if "city" in df.columns:
    df["city"] = df["city"].replace({
        "Banglore": "Bangalore",
        "Bengaluru": "Bangalore",
        "Hyd": "Hyderabad",
        "Del": "Delhi",
        "New Delhi": "Delhi"
    })

if "category" in df.columns:
    df["category"] = df["category"].replace({
        "Govt": "Government",
        "Edu": "Education",
        "Ecommerce": "E-Commerce",
        "E Commerce": "E-Commerce"
    })


In [7]:
# =========================================================
# STEP 6: FIX DATA TYPES
# =========================================================

if "year" in df.columns:
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

if "day" in df.columns:
    df["day"] = pd.to_numeric(df["day"], errors="coerce").astype("Int64")

if "amount_lost_inr" in df.columns:
    df["amount_lost_inr"] = pd.to_numeric(df["amount_lost_inr"], errors="coerce")


In [8]:
# =========================================================
# STEP 7: HANDLE MISSING VALUES
# =========================================================

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("Unknown")

In [9]:
# =========================================================
# STEP 8: HANDLE OUTLIERS
# =========================================================

if "amount_lost_inr" in df.columns:
    q1 = df["amount_lost_inr"].quantile(0.25)
    q3 = df["amount_lost_inr"].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    df["amount_lost_inr"] = df["amount_lost_inr"].clip(
        lower=lower_limit,
        upper=upper_limit
    )


In [10]:
# =========================================================
# STEP 9: FEATURE ENGINEERING
# =========================================================

if "amount_lost_inr" in df.columns:
    df["amount_lost_category"] = pd.qcut(
        df["amount_lost_inr"],
        q=3,
        labels=["Low", "Medium", "High"],
        duplicates="drop"
    )

    high_value_threshold = df["amount_lost_inr"].quantile(0.75)

    df["is_high_value_attack"] = np.where(
        df["amount_lost_inr"] >= high_value_threshold,
        1,
        0
    )

if "day" in df.columns:
    df["day_range"] = pd.cut(
        df["day"],
        bins=[0, 10, 20, 31],
        labels=["Early Month", "Mid Month", "Late Month"],
        include_lowest=True
    )

if "year" in df.columns and "day" in df.columns:
    df["case_date"] = pd.to_datetime(
        df["year"].astype(str) + "-01-" + df["day"].astype(str),
        errors="coerce"
    )


In [11]:
# =========================================================
# STEP 10: FINAL VALIDATION
# =========================================================

print("\nFinal Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nFinal Data Types:")
print(df.dtypes)

print("\nFinal Preview:")
print(df.head())



Final Shape: (1200, 10)

Missing Values:
year                    0
day                     0
amount_lost_inr         0
incident_type           0
city                    0
category                0
amount_lost_category    0
is_high_value_attack    0
day_range               0
case_date               0
dtype: int64

Duplicate Rows: 0

Final Data Types:
year                             Int64
day                              Int64
amount_lost_inr                  int64
incident_type                   object
city                            object
category                        object
amount_lost_category          category
is_high_value_attack             int64
day_range                     category
case_date               datetime64[ns]
dtype: object

Final Preview:
   year  day  amount_lost_inr incident_type       city     category  \
0  2022   22            86530   Data Breach  Bangalore    Corporate   
1  2023   21           231983      Phishing    Kolkata  Educational   
2  2021    6  

In [12]:
df.to_csv("cleaned_cybercrime_dataset.csv", index=False)
print("Dataset exported successfully")

Dataset exported successfully


In [13]:
from sqlalchemy import create_engine
import urllib

# Your SQL Server details
server = r"JYOTHSNA"   # Use raw string for backslash
database = "cybercrime_analytics_db"            # Your database name
driver = "ODBC Driver 17 for SQL Server"   # Must be installed on your machine

# Build connection string for Windows Authentication
connection_string = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

# Create engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={connection_string}")

print("✔ Successfully connected to SQL Server (Windows Authentication)!")

✔ Successfully connected to SQL Server (Windows Authentication)!


In [14]:
table_name = "cyber_attacks"   # SQL table name you want to create

df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"✔ Data successfully loaded into table '{table_name}' in database 'cybercrime_analytics_db'.")

C:\Users\jyoth\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


✔ Data successfully loaded into table 'cyber_attacks' in database 'cybercrime_analytics_db'.
